# 📌 Extracción


In [ ]:
# Importamos las librerías que usará el programa y leemos el JSON inicial
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import requests as rq

telecom_base = rq.get("https://raw.githubusercontent.com/ingridcristh/challenge2-data-science-LATAM/refs/heads/main/TelecomX_Data.json").json()
pd.DataFrame(telecom_base).head()


In [ ]:
pd.DataFrame(telecom_base).info()


# 🔧 Transformación


In [ ]:
# Normalizamos el DataFrame telecom
telecom = pd.json_normalize(telecom_base)
telecom.head()


In [ ]:
# Vemos los tipos de valores de las columnas
telecom.info()


In [ ]:
# Vemos qué valores tienen las columnas (algunas tienen muchos valores distintos, pero la mayoría tiene unos pocos valores únicos)
for nombre_columna in telecom.columns.values.tolist():
  print(f"{nombre_columna}: {telecom[nombre_columna].unique()}")


In [ ]:
# Imprimimos las filas donde 'phone.PhoneService' es 'No' y 'phone.MultipleLines' es distinto de 'No phone service' (Spoiler: no hay ninguna, aoarece un DataFrame con 0
# filas)
print(telecom[(telecom['phone.PhoneService'] == 'No') & (telecom['phone.MultipleLines'] != 'No phone service')])


In [ ]:
# Imprimimos las filas donde 'internet.InternetService' es 'No' y al menos una de las otras filas de 'internet' (OnlineSecurity, OnlineBackup, DeviceProtection,
# TechSupport, StreamingTV, o StreamingMovies) es distinto de 'No internet service' (Spoiler: tampoco hay ninguna)
print(telecom[(telecom['internet.InternetService'] == 'No') & (
    (telecom['internet.OnlineSecurity'] != 'No internet service') |
    (telecom['internet.OnlineBackup'] != 'No internet service') |
    (telecom['internet.DeviceProtection'] != 'No internet service') |
    (telecom['internet.TechSupport'] != 'No internet service') |
    (telecom['internet.StreamingTV'] != 'No internet service') |
    (telecom['internet.StreamingMovies'] != 'No internet service')
)])


In [ ]:
telecom[telecom['Churn'] == ''] # Vemos las filas donde Churn es vacío, y visualizamos el número de filas


In [ ]:
telecom[telecom['account.Charges.Total'] == ' ']


In [ ]:
# Vemos las filas en las que el plazo de contrato (en meses) sea 0. Son precisamente las mismas filas sin monto total registrado
telecom[telecom['customer.tenure'] == 0]


In [ ]:
# Eliminar filas con Churn nulo (un dato crítico)
telecom = telecom[telecom['Churn'] != ''].copy()

# Actualizar PhoneService según tenga líneas múltiples, una única línea o ninguna; y eliminar la columna redundante MultipleLines
telecom.loc[telecom['phone.MultipleLines'] == 'Yes', 'phone.PhoneService'] = 'Multiple lines'
telecom.loc[telecom['phone.PhoneService'] == 'Yes', 'phone.PhoneService'] = 'One line'
telecom.drop(columns=['phone.MultipleLines'], inplace=True)

# Convertir la columna SeniorCitizen en booleano
telecom['customer.SeniorCitizen'] = telecom['customer.SeniorCitizen'].astype(bool)

# Convertir columnas de strings instanciados como 'object' en instancias de 'string'
for nombre_columna in ['customerID', 'customer.gender', 'phone.PhoneService', 'internet.InternetService', 'account.Contract', 'account.PaymentMethod']:
    telecom[nombre_columna] = telecom[nombre_columna].astype('string')

# Convertir columnas de strings con valores Yes/No en booleanos (True/False)
for nombre_columna in ['Churn', 'customer.Partner', 'customer.Dependents', 'account.PaperlessBilling']:
    telecom[nombre_columna] = telecom[nombre_columna].map({'Yes': True, 'No': False}).astype(bool) # Mapea cada columna a una de booleanos

# Convertir las columnas de strings asociadas al servicio de internet (con valores Yes/No/No internet service) en booleanos
for nombre_columna in ['internet.OnlineSecurity', 'internet.OnlineBackup', 'internet.DeviceProtection', 'internet.TechSupport', 'internet.StreamingTV', 'internet.StreamingMovies']:
    telecom[nombre_columna] = telecom[nombre_columna].map({'Yes': True, 'No': False, 'No internet service': False}).astype(bool)

# Parsear Charges.Total a float, rellenando los n/a con ceros
telecom['account.Charges.Total'] = pd.to_numeric(telecom['account.Charges.Total'], errors='coerce').fillna(0.0)

# Por último, vemos de nuevo los tipos de valores de las columnas
telecom.info()


In [ ]:
telecom.head()


In [ ]:
# Paso opcional: Vamos a traducir los nombres de las columnas
telecom = telecom.rename(columns = {
    'customerID': 'ID',
    'Churn': 'Cancelación (churn)',
    'customer.gender': 'Género',
    'customer.SeniorCitizen': 'Jubilado (+60 años)',
    'customer.Partner': 'Con pareja',
    'customer.Dependents': 'Dependientes',
    'customer.tenure': 'Antigüedad (meses)',
    'phone.PhoneService': 'Servicio telefónico',
    'internet.InternetService': 'Servicio de internet',
    'internet.OnlineSecurity': 'Seguridad en línea',
    'internet.OnlineBackup': 'Copia de seguridad en línea',
    'internet.DeviceProtection': 'Protección de dispositivos',
    'internet.TechSupport': 'Soporte técnico',
    'internet.StreamingTV': 'Streaming TV',
    'internet.StreamingMovies': 'Streaming de películas',
    'account.Contract': 'Tipo de contrato',
    'account.PaperlessBilling': 'Factura electrónica',
    'account.PaymentMethod': 'Método de pago',
    'account.Charges.Monthly': 'Cuentas mensuales',
    'account.Charges.Total': 'Cobro total'
})

# Mapeamos las columnas de strings cuyos nombres siguen en inglés
telecom['Género'] = telecom['Género'].replace({'Female': 'Femenino', 'Male': 'Masculino'})
telecom['Servicio telefónico'] = telecom['Servicio telefónico'].replace({'No': 'No', 'One line': 'Una línea', 'Multiple lines': 'Múltiples líneas'})
telecom['Servicio de internet'] = telecom['Servicio de internet'].replace({'No': 'No', 'DSL': 'DSL', 'Fiber optic': 'Fibra óptica'})
telecom['Tipo de contrato'] = telecom['Tipo de contrato'].replace({'One year': 'Un año', 'Month-to-month': 'Mes a mes', 'Two year': 'Dos años'})
telecom['Método de pago'] = telecom['Método de pago'].replace({
    'Mailed check': 'Cheque enviado por correo',
    'Electronic check': 'Cheque electrónico',
    'Credit card (automatic)': 'Tarjeta de crédito (automático)',
    'Bank transfer (automatic)': 'Transferencia bancaria (automático)'
})

# El otro paso opcional: Vamos a crear la columna 'Cuentas diarias' indicando el pago diario promedio de cada cliente a partir de la facturación mensual
telecom['Cuentas diarias'] = (telecom['Cuentas mensuales'] / 30).round(2)

telecom.head()


# 📊 Carga y análisis


In [ ]:
# Análisis descriptivo
telecom.describe()


In [ ]:
# Distribución de evasión
vector_cancelacion = telecom['Cancelación (churn)'].map({
    True: 'Canceló',
    False: 'No canceló'
}).astype('string')
plt.figure(figsize=(8, 6))

ax = vector_cancelacion.value_counts().plot(kind='bar')
plt.title('Distribución de Evasion de Clientes')
plt.xlabel('Cancelamiento')
plt.ylabel('Cantidad de Clientes')
plt.xticks(rotation=0)

plt.ylim(0, vector_cancelacion.value_counts().max() * 1.15)

for i, percentage in enumerate((vector_cancelacion.value_counts() / len(vector_cancelacion)) * 100):
    ax.text(i, vector_cancelacion.value_counts().iloc[i] + 50, f'{percentage:.1f}% ({vector_cancelacion.value_counts().iloc[i]})', ha='center', va='bottom')

plt.tight_layout()
plt.show()


In [ ]:
# Recuento de evasión por variables categóricas
# En principio, la variable categórica será el género
px.histogram(telecom, x = 'Género', text_auto = True, color = 'Cancelación (churn)', barmode = 'group')


In [ ]:
# Ahora el recuento será por el tipo de contrato
px.histogram(telecom, x = 'Tipo de contrato', text_auto = True, color = 'Cancelación (churn)', barmode = 'group')


In [ ]:
# Ahora según el método de pago
px.histogram(telecom, x = 'Método de pago', text_auto = True, color = 'Cancelación (churn)', barmode = 'group')


In [ ]:
# Según la antigüedad
px.histogram(telecom, x = 'Antigüedad (meses)', text_auto = True, color = 'Cancelación (churn)', barmode = 'group')


In [ ]:
# Según la edad
px.histogram(telecom, x = 'Jubilado (+60 años)', text_auto = True, color = 'Cancelación (churn)', barmode = 'group')


In [ ]:
# Y según el tipo de servicio de internet con el que se cuente
px.histogram(telecom, x = 'Servicio de internet', text_auto = True, color = 'Cancelación (churn)', barmode = 'group')


In [ ]:
# Conteo de evasión (cancelación/churn) por costos de servicio por mes

fig1 = px.histogram(telecom, x='Cuentas mensuales', color='Cancelación (churn)', barmode='group',
                   title='Distribución de cancelación por costos mensuales', text_auto=True)
display(fig1)


In [ ]:
# Conteo de evasión por cobro total
fig2 = px.histogram(telecom, x='Cobro total', color='Cancelación (churn)', barmode='group',
                   title='Distribución de Cancelación por Cobro Total', text_auto=True)
display(fig2)


# 📄Informe final
### Introducción
La empresa Telecom X está sufriendo cancelaciones masivas de sus servicios por clientes. En este informe, se analizarán los datos de dicha empresa, relativos a qué clientes cancelaron los servicios y cuáles no, y a otros factores (como género, edad, si cuentan con servicio de teléfono y de internet, subservicios de internet con los que cuenten, e información relativa al tipo de pago realizado), y se estudiarán las potenciales causas o factores relacionados a que muchos clientes cancelen el servicio.

### Desarrollo
Podemos ver, a partir de los gráficos arriba insertados, los factores que impactan en la tasa de cancelaciones que está sufriendo Telecom X, y cómo se distribuye el nivel de cancelaciones.

#### Carga y transformación iniciales
En primer lugar, se importó el JSON inicial desde una URL, y se lo normalizó. Por fortuna, la normalización no incrementó la cantidad inicial de filas de la base de datos, así que el margen de cambios que habría que realizar no se demostró preocupante, aunque sí era momento de hacer cambios: la mayor parte de las columnas eran 'object' en vez de 'string' o 'float64'.

En las columnas 'phone.PhoneService' y 'phone.MultipleLines' se refleja, respectivamente, si un cliente dado tiene o no servicio telefónico, y si tiene múltiples líneas ('Yes'), si no tiene múltiples líneas pero cuenta con servicio telefónico ('No'), o si no sólo no tiene múltiples líneas sino que ni siquiera cuenta con servicio telefónico ('No phone service'). Verifiqué que en las filas en las que un cliente no cuenta con servicio telefónico, se indica adecuadamente también en la columna phone.MultipleLines que no cuenta con servicio telefónico; es decir, no hay inconsistencias. Y en las columnas de 'internet', como bien pude verificar similarmente, se refleja en la columna InternetService si un cliente carece de servicio de internet, y si carece de ello, en las otras columnas de 'internet' el valor no es 'Yes' o 'No' sino 'No internet service'. No obstante, se trata de múltiples dependencias complejas, por lo que opté por no intentar colapsar toda esa información en una sola columna, como sí hice con las columnas de 'phone' (como relataré luego). Luego, se analizaron dos cuestiones sí preocupantes: Por un lado, lo más grave, hay algunas filas en las que el atributo 'Churn' es vacío (vale ''). Dado que es el atributo de principal interés, que no se sepa si un cliente canceló o no es un problema riesgoso para el análisis, aunque alegremente, son pocas las filas donde sucede. Por otro lado, aunque fuera fácil de resolver, la columna de cargos totales 'account.Charges.Total' era vacía en unas 11 filas. Aunque tampoco es agradable que no se sepa el monto total en algunas filas, es fácil de resolver: rellenando con ceros. Además, me encontré con una cuestión analítica entendible lógicamente:

Después, se refleja el proceso de limpieza de los datos y los tipos de éstos. Luego, para resolver la dependencia entre las columnas de 'phone', decidí resolverlo reescribiendo las casillas de phone.PhoneService con valor 'Yes', reescribiéndolas a 'Multiple lines' o 'One line' según el valor de la casilla de la misma fila de 'phone.MultipleLines', y posteriormente eliminé la columna 'phone.MultipleLines' para eliminar la redundancia entre ambas columnas y porque dicha columna se volvió inútil: 'phone.PhoneService' ahora reflejaba toda la información necesaria.

Luego, mapeé la columna de customer.SeniorCitizen como una columna de booleanos (True/False), dado que ya era una columna de enteros y éstos valían sólo 0 (el cliente no era de 60 o más años) o 1 (el cliente sí lo era), y el mapeo a una columna de booleanos era, por lo tanto, automático: 0 equivalía a False y 1 a True. Después, procesé gran parte de las columnas que estaban señalizadas como instancias de 'object' pero realmente eran columnas de 'strings'. Más precisamente, procesé las columnas 'customerID', 'customer.gender', 'phone.PhoneService', 'internet.InternetService', 'account.Contract', y 'account.PaymentMethod'. Las parseé amigablemente como strings, para facilitar su tratamiento. Después, procesé las columnas 'Churn', 'customer.Partner', 'customer.Dependents', y 'account.PaperlessBilling': todas esas columnas consistían también en instancias de 'object' pero que realmente eran strings, pero esos strings valían sólo 'Yes' o 'No'; así que decidí convertir esas columnas en columnas de booleanos, donde 'Yes' era reemplazado por True, y 'No' por False. A continuación, mapeé las columnas de 'internet' (salvo internet.InternetService, que la mapeé simplemente como String) como booleanos, reemplazando los 'Yes' por True, y los 'No' y los 'No internet service' como False; el valor 'No internet service' era una redundancia que indica que el cliente ni siquiera cuenta con el servicio básico de internet. Finalmente, convertí la columna de cobro total, consistente de strings instanciados como 'object' pero consistentes de números flotantes, de los cuales sólo unos 11 tenían valor vacío, en números float, reemplazando los NaNs por ceros, de modo que se convirtió en una columna de floats (float64) sin NaNs.

Después de dicho proceso de transformación, cambié el nombre de las columnas del DataFrame a nombres en español inteligibles, y traduje al español los valores de las columnas cuyos nombres ahora eran 'Género', 'Servicio telefónico', 'Servicio de internet', 'Tipo de contrato', y 'Método de pago': todas estas eran columnas de strings pero cuyos valores eran textos en inglés hasta entonces. Y al final, creé la columna 'Cuentas diarias', consistente en el valor que aproximadamente paga un cliente por día, y calculado a partir del valor de la columna de nombre actual 'Cuentas mensuales' (el valor de 'Cuentas mensuales' dividido por 30 y redondeado a 2 cifras decimales).

#### Carga y análisis de los datos
Una vez procesados los datos, procedí al análisis estadístico. Inicié con un análisis descriptivo, en el que generé una tabla calculando valores estadísticos de interés como la media, mediana, moda, máximo y mínimo de las columnas numéricas del DataFrame, que son la antigüedad en meses del cliente, y el cobro diario, mensual y total; tales valores quedaron reflejados en una tabla allí visible. Allí puede contemplarse que, entre otras cosas, la media de la antigüedad de los clientes es 32.37, ligeramente por arriba de 29, el valor de la mediana, lo cual evidencia un sesgo positivo o corrimiento de la media hacia valores más altos: muchos clientes son más antiguos, lo cual "tira" hacia arriba el promedio. Además, la desviación estándar es de 24.56, un valor muy alto relativamente en comparación con la media, evidenciando la gran variabilidad de la antigüedad en meses de los clientes, que varía de 0 a 72.

Además, la media (64.76) de las cuotas mensuales es inferior a la mediana (70.35), lo cual sugiere que la mayoría de clientes está en un plan de costo medio-alto (probablemente con fibra, y/o servicios adicionales y múltiples líneas telefónicas, por ejemplo), y el promedio se ve reducido por un grupo menor con planes más baratos (el costo mensual mínimo es de 18.25).

Y respecto al cobro total, la media es alrededor de 2279.74, mucho mayor que la mediana, de 1394.55, lo cual indica que una mayoría de clientes pagó montos muy baratos, pero hay un grupo de veteranos que pagó montos altísimos (hasta 8684.8, que es el valor máximo); esto hace que el costo total promedio no sea adecuadamente representativo del cliente típico.

A su vez, la variabilidad de las cuotas mensuales es de un valor elevado: 30.09. Esto indica que los servicios de Telecom X no tienen un precio estándar y el precio promedio no es confiable: hay clientes que pagan por mes desde 18 dolares hasta 118.75 (servicios completos con extensos beneficios). Esto se ve reforzado aún más por la desviación de las cuotas totales: 2266.79, valor casi idéntico al de la media, lo que convierte a las cuotas totales de los clientes en la variable más desequilibrada. Esto indica la gran variabilidad de las cuotas totales, yendo de 0 dolares a 8684.80, y se debe a una causa visible en el gráfico posterior, aquel que agrupa las cantidades de clientes que cancelaron y que no según las cuotas totales: un gran porcentaje, quizás la mayoría, de clientes, pagaron precios no mayores a 100 dolares; la moda de las cuotas mensuales de los clientes también es cercana a 0.

Por último, para el costo diario, la media (2.16) es menor que la mediana (2.34), lo cual indica que la mayor parte de los clientes está pagando, por día, precios superiores al promedio diario, es decir, están concentrados en los planes más costosos o con servicios adicionales, mientras que una minoría con planes muy baratos (con un costo diario de incluso 0.61) reduce la media. Y la desviación estándar es de 1.003, cercano a la mitad de la media. Esto indica una variabilidad pronunciada de los precios diarios: hay quienes pagan desde 0 dolares a 3.96 por día, reforzando también lo indicado sobre la variabilidad de los costos mensuales.

A continuación, calculé la distribución de evasión de los clientes, con un diagrama de barras que calcula cuántos clientes cancelaron, cuántos no, y qué porcentaje del total de clientes contemplados abarca cada conjunto de clientes. Podemos visualizar que 5174 clientes (73,5%) no cancelaron la suscripción, mientras que 1869 (26,5%) sí lo hicieron. La cantidad de clientes que cancelaron está lejos de ser mayoritaria, pero sí supera un cuarto del total, y es una cifra preocupante para una empresa como Telecom X.

#### Conteo de evasión por variables numéricas
Por último, hice una serie de gráficos de barras basados en la relación en las tasas de cancelación con otros factores como el género o el tipo de contrato. He visto que en ambos géneros hay una leve variabilidad entre las cantidades de mujeres y de hombres que mantienen y cancelan el servicio: unas entre 2500 y 2600 personas, de cada género, eligen mantenerlo, mientras otras 930-940 lo abandonan; esto refleja la irrelevancia del género en el abandono del servicio.

El tipo de contrato refleja un factor más interesante: gran parte de los clientes acepta un contrato de un año y otra gran parte uno de dos años, y son muy pocos los que aceptan contratos de uno o dos años y luego los cancelan (poco más de 200, en conjunto). Empero, una mayoría de clientes elige el contrato de mes a mes, y si bien los clientes que eligen este tipo de contrato y no cancelan son la mayoría (pasando no por demasiado a los que eligieron los otros tipos de contrato y mantienen el servicio), la inmensa mayoría de los clientes que sí cancelaron el servicio pagaron un contrato de mes a mes. Los clientes con contrato de mes a mes tienen un compromiso a corto plazo y la 'puerta abierta' para irse en cuanto lo deseen, lo cual hace que se vayan.

El método de pago es un factor también crítico: la mayoría de los clientes opta por el método de pago electrónico, pero, mientras que las cantidades de clientes que eligen cualquiera de los cuatro métodos posibles de pago y no abandonan el servicio no varían mucho, la gran mayoría de los clientes que cancelan el servicio eligieron el pago electrónico. Podría haber errores en el procesamiento de estos pagos, o simplemente ser el método preferido de los clientes que no quieren domiciliar su cuenta.

Midiendo la antigüedad de los clientes que cancelaron y que no, entre los clientes que mantienen el servicio hay dos medias en los meses 0-1 (244 clientes nuevos que mantienen el servicio) y en los meses 72-73 , la mayor antigüedad registrada en meses (359 clientes veteranos que no cancelan). Mientras que, con respecto a los clientes que sí cancelan, hay un sesgo positivo extremo, con la media y la mediana coincidiendo en los meses 0-1: 380 clientes nuevos cancelaron, superando por mucho a los clientes de los meses siguientes. Esto refleja el fracaso de la empresa en la 'bienvenida': Los clientes prueban el servicio y, si no les convence el primer mes, se van de inmediato. A medida que pasan los meses, la probabilidad de que se vayan cae drásticamente (va tendiendo drásticamente al decrecimiento, aunque por momentos crezca levemente como, por ejemplo, en los meses 22-25).

Si miramos los clientes no jubilados y jubilados, podemos ver que la mayoría son no jubilados, y que si bien los no jubilados que no cancelan superan por mucho a los no jubilados que sí, la cantidad de estos últimos sigue siendo preocupante, siendo de quizás alrededor de un cuarto de la cantidad de no jubilados que no cancelaron. Entre los jubilados, si bien son menos, la tasa de fuga es muchísmo más altá: si bien la mayoría de jubilados, 666, mantiene el servicio, unos 476 decide abandonarlo, suponiendo más de dos tercios de la cantidad anterior, y cerca del 42% del total de jubilados, que es un porcentaje peligroso.

El contar o no con servicio de internet y el tipo de servicio también influye importantemente. La mayoría de los clientes elige fibra óptica, y la mayoría de los clientes que mantienen el servicio eligen DSL, si bien las cantidades de clientes que eligieron fibra o no cuentan con internet y no cancelaron el servicio también son similarmente elevadas. Pero, mientras que los clientes sin internet o con DSL que cancelaron el servicio son minoritarias, la mayor parte de los clientes que lo cancelaron eligieron fibra óptica. Esto podría indicar que el servicio de fibra es muy caro o que está teniendo problemas de estabilidad técnica.

Con respecto a los costos mensuales, la mayor parte de la fuga se concentra entre los que pagaron planes de entre 70 y 110 dólares mensuales, reforzando la hipótesis de que los planes premium son los que presentan fallos. Por otro lado, el pico de clientes que mantienen el servicio es entre 0 y 20 mensuales, lo que revela el éxito de los planes baratos. Y por último, viendo los costos totales, hay grandes sesgos positivos: el pico de los clientes que no aceptaron pagaron entre 0 y 300 en total, mientras que una cantidad similarmente grande pagó menos, mientras que el pico de clientes que no abandonaron pagaron como mucho unos 100 en total.

### Conclusión
Precisamente, los tipos de pago, de contrato y de conexión a internet son los factores que más influyen en el abandono, la antigüedad en meses es el mayor reflector del fracaso del servicio con quienes intentan hacer uso por primera vez del mismo, y la tasa de decepción de los jubilados con el sistema es de especial preocupación. Por otro lado, las variabilidades de las antigüedades y las cuotas diarias, mensuales y totales de los clientes reflejan que la base de datos no es uniforme: hay desde clientes que pagan servicios básicos a servicios premium muy costosos, y desde clientes que se anotan masivamente y abandonan (o que se vieron convencidos y se quedaron, aunque sigue siendo su primer mes) hasta veteranos con más de dos años usando los servicios de Telecom X.

Para concluir mi análisis, percibo que la mayor fuga de clientes se produce entre los que pagan los servicios más costosos, más precisamente entre los que pagan fibra óptica de mes a mes con cheque electrónico, y mucho más preponderantemente en el primer mes de servicio, a la vez que más de un 40% de los jubilados se sienten decepcionados.

Como recomendaciones para intentar solucionar el problema, sugiero fuertemente que se revise de inmediato el funcionamiento de la red de fibra de Telecom X y los sistemas encargados del pago electrónico a los fines de solucionar potenciales fallas y problemas de conectividad, procesamiento de pagos, funcionamiento de los dispositivos y otros factores; y que se revise y, si es necesario, se ajuste, el precio de los pagos de mes a mes, a los fines de que quienes opten por este tipo de servicio no se encuentren con un pago exarcebadamente alto. Como medida adicional, se podría hacer encuestas (pueden ser anónimas) a los clientes para estudiar otros factores específicos que puedan hacer que estén disgustados con el servicio, en particular sectores de clientes como los jubilados. La tasa de cancelaciones en los primeros meses es principalmente debido a la mala impresión que está dando actualmente el servicio de Telecom X a quienes intentan hacer uso de éste por primera vez, por lo cual resolver las cuestiones anteriormente planteadas mejorará con seguridad la imagen de dicho servicio ante los nuevos clientes.

Como detalle de interés, considero que si Telecom X llegase a resolver exitosamente las fallas en el sistema, esto podría hacer que mucha más gente desee hacer uso del servicio, lo cual aumentará masivamente la variabilidad de las antigüedades de los clientes, dado que mucha más gente deseará hacer los servicios de la empresa, viéndose mucho más a gusto y básicamente 'llegando para quedarse'.
